# Week 2-3 Sold Dataset EDA

This notebook documents the Week 2-3 exploratory data analysis workflow for the CRMLS sold dataset.

Goals:
- Inspect rows, columns, and data types
- Review unique property types and Residential filtering logic
- Calculate missing counts and missing percentages
- Flag columns with more than 90% missing values
- Separate market analysis fields from metadata fields
- Review numeric distributions and outliers
- Save reports and a filtered analysis dataset

## 1. Import Packages And Set Paths

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REPORT_DIR = PROJECT_ROOT / "data" / "reports" / "week2_3_sold_eda"
FIGURE_DIR = REPORT_DIR / "figures"

REPORT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

SOLD_FILE = PROCESSED_DIR / "crmls_sold_combined_residential_202401_202606.csv"
SOLD_FILE

## 2. Load The Sold Dataset

This file was created in Week 1 by concatenating monthly sold CSVs and filtering to `PropertyType == "Residential"`.

In [ ]:
sold = pd.read_csv(SOLD_FILE, low_memory=False)

# The combined Week 1 file is already filtered to Residential.
# Keep a separate name because the later EDA cells use sold_residential.
sold_residential = sold.copy()

print(f"Rows: {sold_residential.shape[0]:,}")
print(f"Columns: {sold_residential.shape[1]:,}")
sold_residential.head()


## 3. Inspect Structure

In [ ]:
sold.columns

In [ ]:
dtypes_summary = sold.dtypes.astype(str).reset_index()
dtypes_summary.columns = ["column", "dtype"]
dtypes_summary

## 4. Review Property Types

Because this processed file is already Residential-filtered, we expect only Residential rows here. If you need Residential vs. other property type share, use the raw monthly sold files or save an unfiltered combined sold dataset.

In [ ]:
property_type_counts = sold["PropertyType"].fillna("Missing").value_counts(dropna=False)
property_type_counts

## 5. Missing Value Analysis

In [ ]:
missing_report = pd.DataFrame({
    "column": sold_residential.columns,
    "missing_count": sold_residential.isna().sum().values,
})

missing_report["missing_pct"] = (
    missing_report["missing_count"] / len(sold_residential) * 100
).round(2)
missing_report["over_90_pct_missing"] = missing_report["missing_pct"] > 90
missing_report = missing_report.sort_values("missing_pct", ascending=False)

missing_report.head(25)

In [ ]:
high_missing_columns = missing_report[missing_report["over_90_pct_missing"]]
high_missing_columns

## 6. Market Analysis Fields Vs Metadata Fields

This is a first-pass field classification. Core market fields should usually be retained even if they have some missing values.

In [ ]:
core_market_fields = {
    "ListingKey", "ListingId", "CloseDate", "ClosePrice", "ListPrice",
    "OriginalListPrice", "PropertyType", "PropertySubType", "LivingArea",
    "LotSizeAcres", "BedroomsTotal", "BathroomsTotalInteger", "DaysOnMarket",
    "YearBuilt", "CountyOrParish", "City", "PostalCode", "Latitude",
    "Longitude", "ListingContractDate", "PurchaseContractDate",
}

metadata_fields = {
    "ListingKeyNumeric", "ListAgentEmail", "ListAgentFirstName",
    "ListAgentLastName", "ListAgentFullName", "CoListAgentFirstName",
    "CoListAgentLastName", "CoBuyerAgentFirstName", "BuyerAgentMlsId",
    "BuyerAgentFirstName", "BuyerAgentLastName", "ListOfficeName",
    "BuyerOfficeName", "CoListOfficeName", "BuyerOfficeAOR",
    "BuyerAgentAOR", "ListAgentAOR", "OriginatingSystemName",
    "OriginatingSystemSubName", "MlsStatus",
}

field_classification = []
for column in sold_residential.columns:
    if column in metadata_fields:
        category = "metadata"
    elif column in core_market_fields:
        category = "market_analysis_core"
    else:
        category = "market_analysis_other"
    field_classification.append({"column": column, "field_category": category})

field_classification = pd.DataFrame(field_classification)
field_classification

## 7. Numeric Distribution Summary

In [ ]:
numeric_fields = [
    "ClosePrice", "ListPrice", "OriginalListPrice", "LivingArea",
    "LotSizeAcres", "BedroomsTotal", "BathroomsTotalInteger",
    "DaysOnMarket", "YearBuilt",
]

numeric_summary_rows = []

for field in numeric_fields:
    series = pd.to_numeric(sold_residential[field], errors="coerce")
    percentiles = series.quantile([0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99])
    numeric_summary_rows.append({
        "field": field,
        "count": int(series.count()),
        "missing_count": int(series.isna().sum()),
        "min": series.min(),
        "max": series.max(),
        "mean": series.mean(),
        "median": series.median(),
        "p01": percentiles.loc[0.01],
        "p05": percentiles.loc[0.05],
        "p25": percentiles.loc[0.25],
        "p50": percentiles.loc[0.5],
        "p75": percentiles.loc[0.75],
        "p95": percentiles.loc[0.95],
        "p99": percentiles.loc[0.99],
    })

numeric_summary = pd.DataFrame(numeric_summary_rows)
numeric_summary

## 8. Histograms And Boxplots

In [ ]:
for field in numeric_fields:
    series = pd.to_numeric(sold_residential[field], errors="coerce").dropna()
    if series.empty:
        continue

    p01 = series.quantile(0.01)
    p99 = series.quantile(0.99)
    zoomed_series = series[(series >= p01) & (series <= p99)]
    outside_zoom_count = len(series) - len(zoomed_series)

    fig, axes = plt.subplots(2, 2, figsize=(14, 8))

    axes[0, 0].hist(series, bins=50)
    axes[0, 0].set_title(f"{field} Histogram - Full Range")
    axes[0, 0].set_xlabel(field)
    axes[0, 0].set_ylabel("Count")

    axes[0, 1].boxplot(series, vert=False, showfliers=True)
    axes[0, 1].set_title(f"{field} Boxplot - Full Range")
    axes[0, 1].set_xlabel(field)

    axes[1, 0].hist(zoomed_series, bins=50)
    axes[1, 0].set_title(f"{field} Histogram - 1st to 99th Percentile")
    axes[1, 0].set_xlabel(field)
    axes[1, 0].set_ylabel("Count")

    axes[1, 1].boxplot(zoomed_series, vert=False, showfliers=True)
    axes[1, 1].set_title(f"{field} Boxplot - 1st to 99th Percentile")
    axes[1, 1].set_xlabel(field)

    fig.suptitle(
        f"{field}: full range plus zoomed view. "
        f"Rows outside 1st-99th percentile view: {outside_zoom_count:,}",
        y=1.02,
    )

    plt.tight_layout()
    plt.savefig(FIGURE_DIR / f"{field}_full_and_zoomed_histogram_boxplot.png")
    plt.show()

## 9. Extreme Outlier Review

This uses the IQR rule to identify values outside `Q1 - 1.5 * IQR` and `Q3 + 1.5 * IQR`. These are not automatically removed; they are flagged for later cleaning decisions.

In [ ]:
outlier_rows = []

for field in numeric_fields:
    series = pd.to_numeric(sold_residential[field], errors="coerce").dropna()
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    low_outliers = int((series < lower_bound).sum())
    high_outliers = int((series > upper_bound).sum())
    
    outlier_rows.append({
        "field": field,
        "iqr_lower_bound": lower_bound,
        "iqr_upper_bound": upper_bound,
        "low_outlier_count": low_outliers,
        "high_outlier_count": high_outliers,
        "total_outlier_count": low_outliers + high_outliers,
        "outlier_pct": round((low_outliers + high_outliers) / len(series) * 100, 2),
        "min": series.min(),
        "max": series.max(),
    })

outlier_report = pd.DataFrame(outlier_rows)
outlier_report

## 10. Answer EDA Questions

In [ ]:
close_price = pd.to_numeric(sold_residential["ClosePrice"], errors="coerce")
list_price = pd.to_numeric(sold_residential["ListPrice"], errors="coerce")
days_on_market = pd.to_numeric(sold_residential["DaysOnMarket"], errors="coerce")

valid_price_pairs = sold_residential[close_price.notna() & list_price.notna()].copy()
valid_price_pairs["ClosePriceNumeric"] = pd.to_numeric(valid_price_pairs["ClosePrice"], errors="coerce")
valid_price_pairs["ListPriceNumeric"] = pd.to_numeric(valid_price_pairs["ListPrice"], errors="coerce")

above_list_pct = (valid_price_pairs["ClosePriceNumeric"] > valid_price_pairs["ListPriceNumeric"]).mean() * 100
below_list_pct = (valid_price_pairs["ClosePriceNumeric"] < valid_price_pairs["ListPriceNumeric"]).mean() * 100
at_list_pct = (valid_price_pairs["ClosePriceNumeric"] == valid_price_pairs["ListPriceNumeric"]).mean() * 100

close_date = pd.to_datetime(sold_residential["CloseDate"], errors="coerce")
listing_date = pd.to_datetime(sold_residential["ListingContractDate"], errors="coerce")
purchase_date = pd.to_datetime(sold_residential["PurchaseContractDate"], errors="coerce")

summary_answers = pd.DataFrame([
    {"question": "Average close price", "answer": close_price.mean()},
    {"question": "Median close price", "answer": close_price.median()},
    {"question": "Average Days on Market", "answer": days_on_market.mean()},
    {"question": "Median Days on Market", "answer": days_on_market.median()},
    {"question": "Sold above list price pct", "answer": above_list_pct},
    {"question": "Sold below list price pct", "answer": below_list_pct},
    {"question": "Sold at list price pct", "answer": at_list_pct},
    {"question": "CloseDate before ListingContractDate rows", "answer": int((close_date < listing_date).sum())},
    {"question": "CloseDate before PurchaseContractDate rows", "answer": int((close_date < purchase_date).sum())},
])

summary_answers

In [ ]:
county_median_prices = (
    sold_residential.assign(ClosePriceNumeric=close_price)
    .dropna(subset=["CountyOrParish", "ClosePriceNumeric"])
    .groupby("CountyOrParish")["ClosePriceNumeric"]
    .median()
    .sort_values(ascending=False)
    .reset_index(name="median_close_price")
)

county_median_prices.head(10)

## 11. Save Deliverable Outputs

In [ ]:
filtered_output = PROCESSED_DIR / "crmls_sold_residential_week2_3_filtered_202401_202606.csv"

sold_residential.to_csv(filtered_output, index=False)
dtypes_summary.to_csv(REPORT_DIR / "dtypes_summary.csv", index=False)
property_type_counts.reset_index().rename(columns={"index": "PropertyType", "PropertyType": "row_count"}).to_csv(REPORT_DIR / "property_type_counts.csv", index=False)
field_classification.to_csv(REPORT_DIR / "field_classification.csv", index=False)
missing_report.to_csv(REPORT_DIR / "missing_value_report.csv", index=False)
numeric_summary.to_csv(REPORT_DIR / "numeric_distribution_summary.csv", index=False)
outlier_report.to_csv(REPORT_DIR / "numeric_outlier_report.csv", index=False)
summary_answers.to_csv(REPORT_DIR / "eda_question_answers.csv", index=False)
county_median_prices.to_csv(REPORT_DIR / "county_median_close_prices.csv", index=False)

print("Saved filtered dataset:", filtered_output)
print("Saved reports to:", REPORT_DIR)
print("Saved figures to:", FIGURE_DIR)